# Compare TRACE vs PyPSA-Earth Renewable Datasets

**Objective:** Compare the legacy TRACE renewable data with the new PyPSA-Earth clustered output.

- **TRACE:** External pre-aggregated legacy dataset from the TRACE package. It is the deprecated reference source for the old consolidated structure.
- **PyPSA-Earth:** Newly calculated clustered renewable dataset built from the atlite workflow. It preserves cell-level spatial resolution through region-technology pseudo-generators.

**Primary comparison goals:**
1. Compare installed capacity and energy potential between TRACE and PyPSA-Earth.
2. Visualize capacity-factor distributions and merit-order structure.
3. Compare cluster-level timeseries behavior per region.
4. Keep the stacked region comparison as the main regional capacity overview.
5. Use one consistent visual style across all plots.

In [ ]:
import logging
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import xarray as xr
from IPython.display import display
import plot_helpers

sns.set_theme(style="whitegrid", context="talk", font_scale=1.12)
plt.rcParams.update(
    {
        "figure.figsize": (14, 6),
        "figure.titlesize": 16,
        "font.size": 11,
        "axes.titlesize": 14,
        "axes.labelsize": 12,
        "axes.titleweight": "semibold",
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "legend.fontsize": 10,
        "legend.title_fontsize": 11,
        "axes.labelpad": 8,
    }
)
logging.basicConfig(level=logging.INFO, format="%(message)s")

BASE_DIR = Path("../..")
TRACE_FILE = BASE_DIR / "data/new_renewables_consolidated.nc"
PYPSA_EARTH_FILE = BASE_DIR / "resources/renewables_clustered.nc"
PYPSA_EARTH_META_FILE = BASE_DIR / "resources/clusters_cache.joblib"

SOURCE_STYLES = {
    "TRACE": {"color": "#6B7280", "marker": "o"},
    "PyPSA-Earth": {"color": "#A8A29E", "marker": "s"},
}
TECH_COLORS = {
    "onwind": "#1F77B4",
    "solar": "#D62728",
}

print(f"TRACE:        {'✓' if TRACE_FILE.exists() else '✗'} {TRACE_FILE}")
print(f"PyPSA-Earth:  {'✓' if PYPSA_EARTH_FILE.exists() else '✗'} {PYPSA_EARTH_FILE}")
print(
    f"Metadata:     {'✓' if PYPSA_EARTH_META_FILE.exists() else '✗'} {PYPSA_EARTH_META_FILE}"
)

trace_ds = xr.open_dataset(TRACE_FILE) if TRACE_FILE.exists() else None
pypsa_earth_ds = (
    xr.open_dataset(PYPSA_EARTH_FILE) if PYPSA_EARTH_FILE.exists() else None
)
pypsa_earth_meta = None
if PYPSA_EARTH_META_FILE.exists():
    import joblib

    pypsa_earth_meta = joblib.load(PYPSA_EARTH_META_FILE)

summary_rows = []
for label, ds in [("TRACE", trace_ds), ("PyPSA-Earth", pypsa_earth_ds)]:
    if ds is None:
        continue
    summary_rows.append(
        {
            "source": label,
            "dimensions": dict(ds.sizes),
            "variables": ", ".join(list(ds.data_vars)),
            "regions": int(ds.sizes.get("region", 0)),
            "technologies": int(ds.sizes.get("technology", 0)),
            "classes": int(ds.sizes.get("class", 0)),
            "time_steps": int(ds.sizes.get("time", ds.sizes.get("hour", 0))),
        }
    )

summary_df = pd.DataFrame(summary_rows)
if not summary_df.empty:
    display(summary_df)

if pypsa_earth_meta is not None:
    metadata_df = pd.DataFrame(
        [
            dict(info, cluster_name=name)
            for name, info in pypsa_earth_meta.get("cluster_metadata", {}).items()
        ]
    )
    if not metadata_df.empty:
        metadata_df["technology"] = metadata_df["technology"].astype(str)
        metadata_df["region"] = metadata_df["region"].astype(str)
        display(
            metadata_df[
                [
                    "region",
                    "technology",
                    "cluster_id",
                    "n_buses_consolidated",
                    "total_capacity_mw",
                    "avg_cf",
                ]
            ].head(12)
        )

In [ ]:
print("\n" + "=" * 70)
print("TRACE DATASET DIAGNOSTICS")
print("=" * 70)

trace_diagnostic_rows = []
if trace_ds is not None:
    trace_diagnostic_rows.append(
        {
            "source": "TRACE",
            "dimensions": dict(trace_ds.sizes),
            "regions": int(trace_ds.sizes.get("region", 0)),
            "technologies": int(trace_ds.sizes.get("technology", 0)),
            "sites": int(trace_ds.sizes.get("class", trace_ds.sizes.get("site_id", 0))),
            "time_steps": int(
                trace_ds.sizes.get("time", trace_ds.sizes.get("hour", 0))
            ),
            "variables": ", ".join(list(trace_ds.data_vars)),
        }
    )
    trace_diag_df = pd.DataFrame(trace_diagnostic_rows)
    display(trace_diag_df)

    trace_capacity_stats = []
    for tech_name in ["windonshore", "pvplant"]:
        if "technology" not in trace_ds.coords:
            continue
        if tech_name not in [str(v) for v in trace_ds.technology.values]:
            continue
        try:
            capacity_values = trace_ds["capacity"].sel(technology=tech_name).values
            capacity_values = capacity_values[~np.isnan(capacity_values)]
            trace_capacity_stats.append(
                {
                    "technology": tech_name,
                    "count": int(len(capacity_values)),
                    "min_mw": float(np.min(capacity_values)),
                    "median_mw": float(np.median(capacity_values)),
                    "mean_mw": float(np.mean(capacity_values)),
                    "max_mw": float(np.max(capacity_values)),
                }
            )
        except Exception:
            continue
    if trace_capacity_stats:
        display(pd.DataFrame(trace_capacity_stats))
else:
    print("TRACE file not found.")

print("\n" + "=" * 70)
print("PYPSA-EARTH DATASET DIAGNOSTICS")
print("=" * 70)

pypsa_diagnostic_rows = []
if pypsa_earth_ds is not None:
    pypsa_diagnostic_rows.append(
        {
            "source": "PyPSA-Earth",
            "dimensions": dict(pypsa_earth_ds.sizes),
            "regions": int(pypsa_earth_ds.sizes.get("region", 0)),
            "technologies": int(pypsa_earth_ds.sizes.get("technology", 0)),
            "clusters": int(pypsa_earth_ds.sizes.get("class", 0)),
            "time_steps": int(
                pypsa_earth_ds.sizes.get("time", pypsa_earth_ds.sizes.get("hour", 0))
            ),
            "variables": ", ".join(list(pypsa_earth_ds.data_vars)),
        }
    )
    pypsa_diag_df = pd.DataFrame(pypsa_diagnostic_rows)
    display(pypsa_diag_df)

    pypsa_capacity_stats = []
    for tech_name in ["onwind", "solar"]:
        if tech_name not in [str(v) for v in pypsa_earth_ds.technology.values]:
            continue
        try:
            capacity_values = (
                pypsa_earth_ds["capacity"].sel(technology=tech_name).values.flatten()
            )
            capacity_values = capacity_values[~np.isnan(capacity_values)]
            pypsa_capacity_stats.append(
                {
                    "technology": tech_name,
                    "count": int(len(capacity_values)),
                    "min_mw": float(np.min(capacity_values)),
                    "median_mw": float(np.median(capacity_values)),
                    "mean_mw": float(np.mean(capacity_values)),
                    "max_mw": float(np.max(capacity_values)),
                }
            )
        except Exception:
            continue
    if pypsa_capacity_stats:
        display(pd.DataFrame(pypsa_capacity_stats))
else:
    print("PyPSA-Earth file not found.")

In [ ]:
print("=" * 70)
print("BUILDING COMPARISON TABLES")
print("=" * 70)

source_order = ["TRACE", "PyPSA-Earth"]
tech_order = ["solar", "onwind"]
source_palette = SOURCE_STYLES
tech_palette = TECH_COLORS


def build_trace_rows(ds):
    rows = []
    if ds is None:
        return rows
    for trace_tech_name, compare_tech_name in {
        "windonshore": "onwind",
        "pvplant": "solar",
    }.items():
        if trace_tech_name not in [str(v) for v in ds.technology.values]:
            continue
        try:
            cap_da = ds["capacity"].sel(technology=trace_tech_name)
            if "avg_cf" in ds.data_vars:
                avg_cf_da = ds["avg_cf"].sel(technology=trace_tech_name)
            else:
                avg_cf_da = (
                    ds["capacity_factor"]
                    .sel(technology=trace_tech_name)
                    .mean(dim="time", skipna=True)
                )

            for region_name in ds.region.values:
                region_caps = cap_da.sel(region=region_name).values
                region_avg_cf = avg_cf_da.sel(region=region_name).values
                for class_idx in range(len(region_caps)):
                    capacity_mw = float(region_caps[class_idx])
                    avg_cf = float(region_avg_cf[class_idx])
                    if np.isnan(capacity_mw) or np.isnan(avg_cf):
                        continue
                    rows.append(
                        {
                            "source": "TRACE",
                            "region": str(region_name),
                            "technology": str(compare_tech_name),
                            "cluster_id": int(class_idx),
                            "label": f"{compare_tech_name} {class_idx + 1}",
                            "capacity_mw": capacity_mw,
                            "avg_cf": avg_cf,
                            "generation_potential_mwh": capacity_mw * avg_cf * 8760.0,
                        }
                    )
        except Exception as exc:
            print(f"  Could not read TRACE data for {trace_tech_name}: {exc}")
    return rows


def build_pypsa_rows(ds, meta):
    rows = []
    if meta is not None and meta.get("cluster_metadata") is not None:
        for cluster_name, info in meta.get("cluster_metadata", {}).items():
            rows.append(
                {
                    "source": "PyPSA-Earth",
                    "region": str(info["region"]),
                    "technology": str(info["technology"]),
                    "cluster_id": int(info["cluster_id"]),
                    "label": f"{info['technology']} {int(info['cluster_id']) + 1}",
                    "capacity_mw": float(info["total_capacity_mw"]),
                    "avg_cf": float(info["avg_cf"]),
                    "generation_potential_mwh": float(
                        info["total_capacity_mw"] * info["avg_cf"] * 8760.0
                    ),
                }
            )
        return rows

    if ds is None:
        return rows

    for tech_name in tech_order:
        if tech_name not in [str(v) for v in ds.technology.values]:
            continue
        try:
            cap_da = ds["capacity"].sel(technology=tech_name)
            avg_cf_da = ds["avg_cf"].sel(technology=tech_name)
            for region_name in ds.region.values:
                region_caps = cap_da.sel(region=region_name).values
                region_avg_cf = avg_cf_da.sel(region=region_name).values
                for class_idx in range(len(region_caps)):
                    capacity_mw = float(region_caps[class_idx])
                    avg_cf = float(region_avg_cf[class_idx])
                    if np.isnan(capacity_mw) or np.isnan(avg_cf):
                        continue
                    rows.append(
                        {
                            "source": "PyPSA-Earth",
                            "region": str(region_name),
                            "technology": str(tech_name),
                            "cluster_id": int(class_idx),
                            "label": f"{tech_name} {class_idx + 1}",
                            "capacity_mw": capacity_mw,
                            "avg_cf": avg_cf,
                            "generation_potential_mwh": capacity_mw * avg_cf * 8760.0,
                        }
                    )
        except Exception as exc:
            print(f"  Could not read PyPSA-Earth data for {tech_name}: {exc}")
    return rows


trace_rows = build_trace_rows(trace_ds)
pypsa_rows = build_pypsa_rows(pypsa_earth_ds, pypsa_earth_meta)
comparison_df = pd.concat(
    [pd.DataFrame(trace_rows), pd.DataFrame(pypsa_rows)], ignore_index=True
)
if comparison_df.empty:
    raise ValueError("No comparison data found.")

comparison_df["source"] = pd.Categorical(
    comparison_df["source"], categories=source_order, ordered=True
)
comparison_df["technology"] = pd.Categorical(
    comparison_df["technology"], categories=tech_order, ordered=True
)
comparison_df["region"] = comparison_df["region"].astype(str)
comparison_df["avg_cf"] = comparison_df["avg_cf"].astype(float)
comparison_df["capacity_mw"] = comparison_df["capacity_mw"].astype(float)
comparison_df["generation_potential_mwh"] = comparison_df[
    "generation_potential_mwh"
].astype(float)

comparison_summary_df = (
    comparison_df.groupby(["source", "technology"], as_index=False, observed=True)
    .agg(
        n_clusters=("cluster_id", "size"),
        total_capacity_mw=("capacity_mw", "sum"),
        mean_avg_cf=("avg_cf", "mean"),
        total_generation_potential_mwh=("generation_potential_mwh", "sum"),
    )
    .sort_values(["source", "technology"])
)

display(comparison_summary_df)

### Bubble Plot: Capacity Factor vs Capacity by Source

This plot compares the distribution of cluster capacities and average capacity factors between TRACE and PyPSA-Earth.\nIt reveals whether clustering **preserves the merit-order structure**: low-cost (high CF) clusters should remain distinct and visible as separate bubbles.\nBubble size represents annual generation potential. Log scale on the Y-axis emphasizes smaller but important clusters.



In [ ]:
print("\\nGenerating bubble plots...")

if trace_ds is None or pypsa_earth_ds is None:
    raise ValueError(
        "Both TRACE and PyPSA-Earth datasets are required for the bubble plots."
    )

common_regions = sorted(
    set(str(v) for v in trace_ds.region.values)
    & set(str(v) for v in pypsa_earth_ds.region.values)
)
if not common_regions:
    raise ValueError("No common regions found between TRACE and PyPSA-Earth.")

region = common_regions[0]
print(f"Example region: {region}")

plot_helpers.plot_bubble_region_comparison(region, trace_ds, pypsa_earth_ds, show=True)
print("✓ Bubble plot rendered for TRACE and PyPSA-Earth.")

In [ ]:
plot_helpers.plot_bubble_region_comparison(
    "West_Asia", trace_ds, pypsa_earth_ds, show=True
)

### Interactive Timeseries Comparison: Cluster Capacity Factors Over Time

This plot compares how individual clusters behave over a full year in TRACE vs PyPSA-Earth.
It validates that clustering **preserves temporal patterns** (peak periods, seasonal variation) and the **energy density hierarchy** (line width scales with average CF).
Two stacked panels show each source; separate invocations for solar and onwind allow independent inspection of each technology's cluster dynamics.

In [ ]:
print("\\nGenerating interactive timeseries comparison panels...")

if trace_ds is None or pypsa_earth_ds is None:
    raise ValueError(
        "Both TRACE and PyPSA-Earth datasets are required for the timeseries comparison."
    )

common_regions = sorted(
    set(str(v) for v in trace_ds.region.values)
    & set(str(v) for v in pypsa_earth_ds.region.values)
)
if not common_regions:
    raise ValueError("No common regions found between TRACE and PyPSA-Earth.")

region = common_regions[0]
print(f"Example region: {region}")

print("\\n--- Solar Timeseries ---")
plot_helpers.plot_region_cluster_timeseries(
    region, "solar", trace_ds=trace_ds, pypsa_ds=pypsa_earth_ds, show=True
)

print("\\n--- Onwind Timeseries ---")
plot_helpers.plot_region_cluster_timeseries(
    region, "onwind", trace_ds=trace_ds, pypsa_ds=pypsa_earth_ds, show=True
)

print("✓ Interactive timeseries comparison rendered for both technologies.")

## Stacked Region Comparison: Total Capacity by Cluster Merit Order

This plot stacks cluster capacities per region, ordered by average capacity factor (shaded light → dark).
It shows whether clustering **preserves regional capacity distributions** and **maintains the high-CF "green pockets"** that enable merit-order effects in optimization.
Side-by-side panels compare solar and onwind; border colors distinguish Consolidated vs Clustered sources.

In [ ]:
print("\\nGenerating stacked region comparison by technology...")

# Build cluster comparison dataframe if not already in memory
ds_consolidated = trace_ds if "trace_ds" in dir() and trace_ds is not None else None
ds_clustered = (
    pypsa_earth_ds if "pypsa_earth_ds" in dir() and pypsa_earth_ds is not None else None
)
df_meta = pypsa_earth_meta if "pypsa_earth_meta" in dir() else None

df_cluster_meta = None
if df_meta is not None and "cluster_metadata" in df_meta:
    metadata_rows = []
    for name, info in df_meta.get("cluster_metadata", {}).items():
        row = dict(info)
        row["cluster_name"] = name
        metadata_rows.append(row)
    if metadata_rows:
        df_cluster_meta = pd.DataFrame(metadata_rows)

if "cluster_comparison_df" not in dir():
    cluster_comparison_df = plot_helpers.build_cluster_comparison_df(
        ds_consolidated, ds_clustered, df_cluster_meta
    )
    if cluster_comparison_df.empty:
        raise ValueError(
            "Cluster-level comparison data is required for the stacked region comparison."
        )

plot_helpers.plot_stacked_region_comparison(
    cluster_comparison_df, show=True, save_path="region_stacked_capacity_comparison.png"
)
print(
    "✓ Stacked region comparison rendered and saved as region_stacked_capacity_comparison.png"
)